<div dir="rtl" style="text-align:right; line-height:2">

# پروژه پایانی: خط لوله تحلیل داده‌های زنده سفرهای تاکسی (NYC Taxi)

**معماری کلی پروژه:**

`داده خام TLC → پاک‌سازی (pandas) → HDFS → MapReduce / Hadoop Streaming → Spark (batch) → Kafka → Spark Structured Streaming → داشبورد`

**محیط تست‌شده:** macOS، Python 3.11 (venv)، Java 17، Docker Desktop. ایمیج‌ها: `apache/hadoop:3.3.6` و `apache/kafka:3.7.0`؛ Spark 3.5.1.

**پیش‌نیازها روی سیستم شما:** Docker (با Docker Compose)، Python 3.9+، Java 11 یا 17 (برای PySpark) و اینترنت (برای دانلود داده و پکیج‌ها).

**ساختار پوشه‌ها** (همه‌چیز خودکار ساخته می‌شود):

```
taxi_project/
├── data/        # داده خام و پاک‌سازی‌شده
├── hadoop/      # docker-compose هدوپ + اسکریپت‌های mapper/reducer
├── kafka/       # docker-compose کافکا + producer
└── results/     # خروجی‌های MapReduce
```

> نکته: همه نمودارها برچسب انگلیسی دارند تا نیازی به نصب فونت فارسی برای matplotlib نباشد.

</div>

In [ ]:
%pip install -q pandas pyarrow requests matplotlib pyspark==3.5.1 kafka-python

In [ ]:
from pathlib import Path
import time, subprocess, json, sys

BASE = Path("taxi_project")
for d in ["data", "hadoop/task4", "hadoop/task5", "kafka", "results"]:
    (BASE / d).mkdir(parents=True, exist_ok=True)

MONTH = "2024-01"          # ماه داده TLC
SAMPLE_ROWS = None         # برای تست سریع مثلا 300000 بگذارید؛ None یعنی کل داده
print("ساختار پوشه‌ها آماده شد:", BASE.resolve())

<div dir="rtl" style="text-align:right; line-height:2">

---
# بخش ۱: درک Big Data — پنج V

**بیگ‌دیتا** به مجموعه‌داده‌هایی گفته می‌شود که حجم، سرعت تولید یا تنوع آن‌ها به حدی است که ابزارهای سنتی (مثل یک پایگاه‌داده رابطه‌ای روی یک سرور) نمی‌توانند آن‌ها را به‌صورت کارآمد ذخیره و پردازش کنند. این مفهوم معمولاً با پنج «V» توضیح داده می‌شود:

| V | معنی | مثال واقعی | نمونه در پروژه ما |
|---|------|-----------|-------------------|
| **Volume (حجم)** | مقدار عظیم داده؛ از ترابایت تا پتابایت و بیشتر | فیسبوک/اینستاگرام روزانه ده‌ها ترابایت عکس و ویدیو ذخیره می‌کنند | TLC سالانه صدها میلیون سفر ثبت می‌کند؛ هر ماه حدود ۳ میلیون ردیف |
| **Velocity (سرعت)** | سرعت تولید و نیاز به پردازش داده، گاهی بلادرنگ | تشخیص تقلب بانکی در چند میلی‌ثانیه هنگام تراکنش کارت | استریم سفرها از Kafka و پردازش لحظه‌ای با Spark Streaming |
| **Variety (تنوع)** | قالب‌های گوناگون: ساخت‌یافته، نیمه‌ساخت‌یافته (JSON/XML)، بدون ساختار (متن، تصویر، صوت) | سیستم توصیه‌گر نتفلیکس از لاگ تماشا، متن نقد و تصاویر استفاده می‌کند | Parquet، CSV و پیام‌های JSON در Kafka |
| **Veracity (صحت)** | کیفیت و قابل‌اعتماد بودن داده؛ نویز، تکراری، ناقص، اشتباه | داده حسگرهای IoT اغلب مقدارهای خراب یا گم‌شده دارد | سفرهای با مسافت صفر، مدت منفی یا تاریخ اشتباه که در پاک‌سازی حذف می‌کنیم |
| **Value (ارزش)** | تبدیل داده به بینش و تصمیم کسب‌وکاری | شرکت‌های حمل‌ونقل با تحلیل تقاضا قیمت‌گذاری پویا انجام می‌دهند | شناسایی ساعات و مناطق پرتقاضا برای مدیریت ناوگان |

### کاربردها در صنایع مختلف
- **سلامت:** تحلیل پرونده‌های الکترونیک و ژنوم برای پزشکی شخصی‌سازی‌شده و پیش‌بینی شیوع بیماری.
- **مالی و بانکی:** شناسایی تقلب، امتیازدهی اعتباری، معاملات الگوریتمی.
- **خرده‌فروشی و تجارت الکترونیک:** توصیه محصول (آمازون)، پیش‌بینی تقاضا و مدیریت موجودی.
- **حمل‌ونقل:** تخمین زمان رسیدن (ETA)، مسیریابی ترافیک، قیمت‌گذاری پویا (Uber، اسنپ).
- **مخابرات:** پیش‌بینی ریزش مشتری (Churn) و پایش کیفیت شبکه از لاگ دکل‌ها.
- **تولید و صنعت:** نگهداری پیش‌بینانه ماشین‌آلات با داده حسگرها.
- **انرژی:** شبکه‌های هوشمند برق و پیش‌بینی مصرف.

**جمع‌بندی:** داده‌های سفر تاکسی هر پنج V را دارند و به همین دلیل نمونه آموزشی مناسبی برای کل زنجیره Hadoop ← Spark ← Kafka هستند.

</div>

<div dir="rtl" style="text-align:right; line-height:2">

---
# بخش ۲: دریافت و پاک‌سازی داده (Data Acquisition)

**هدف:** دانلود داده سفرهای Yellow Taxi از سایت TLC شهر نیویورک، پاک‌سازی و ذخیره برای مراحل بعد.

**مراحل پاک‌سازی و دلیل هرکدام:**
1. **حذف مقادیر گم‌شده (NaN)** در ستون‌های کلیدی، چون در تحلیل و Mapper خطا می‌دهند.
2. **حذف ردیف‌های تکراری.**
3. **اعتبارسنجی زمان:** مدت سفر باید بین ۱ دقیقه و ۳ ساعت باشد (سفر منفی یا چندروزه یعنی خطای ثبت).
4. **اعتبارسنجی مسافت:** بین ۰٫۱ تا ۱۰۰ مایل.
5. **اعتبارسنجی کرایه و تعداد مسافر:** کرایه مثبت و تعداد مسافر ۱ تا ۶.
6. **حذف سفرهایی که خارج از ماه مورد نظر هستند** (فایل‌های TLC گاهی تاریخ‌های اشتباه مثل ۲۰۰۲ دارند).
7. **ساخت ستون‌های مشتق‌شده:** `duration_min` و `pickup_date`.

خروجی را به دو شکل ذخیره می‌کنیم: **CSV** (برای Hadoop و Spark) و **Parquet** (فشرده و سریع).

</div>

In [ ]:
import pandas as pd, requests

url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{MONTH}.parquet"
raw_path = BASE / "data" / f"yellow_tripdata_{MONTH}.parquet"

if not raw_path.exists():
    print("در حال دانلود:", url)
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        with open(raw_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
print("حجم فایل خام: %.1f MB" % (raw_path.stat().st_size / 1e6))

df_raw = pd.read_parquet(raw_path)
print("تعداد ردیف خام:", len(df_raw))
df_raw.head()

In [ ]:
print(df_raw.dtypes)
print()
print("مقادیر گم‌شده در هر ستون:")
print(df_raw.isna().sum()[df_raw.isna().sum() > 0])
print()
print(df_raw[["trip_distance", "fare_amount", "passenger_count"]].describe().round(2))

In [ ]:
def clean_trips(df: pd.DataFrame, month: str) -> pd.DataFrame:
    cols = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "passenger_count",
            "trip_distance", "PULocationID", "DOLocationID", "fare_amount", "total_amount"]
    df = df[cols].copy()
    n0 = len(df)

    df = df.dropna()                       # ۱) مقادیر گم‌شده
    df = df.drop_duplicates()              # ۲) تکراری‌ها

    df["duration_min"] = (df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]).dt.total_seconds() / 60
    df = df[(df["duration_min"] >= 1) & (df["duration_min"] <= 180)]          # ۳) مدت
    df = df[(df["trip_distance"] >= 0.1) & (df["trip_distance"] <= 100)]      # ۴) مسافت
    df = df[(df["fare_amount"] > 0) & (df["total_amount"] > 0)]               # ۵) کرایه
    df = df[df["passenger_count"].between(1, 6)]                              # ۵) مسافر

    start = pd.Timestamp(month + "-01")
    end = start + pd.offsets.MonthBegin(1)
    df = df[(df["tpep_pickup_datetime"] >= start) & (df["tpep_pickup_datetime"] < end)]  # ۶) ماه

    df["passenger_count"] = df["passenger_count"].astype(int)
    df["pickup_date"] = df["tpep_pickup_datetime"].dt.strftime("%Y-%m-%d")    # ۷)
    df["duration_min"] = df["duration_min"].round(2)
    df = df.rename(columns={"tpep_pickup_datetime": "pickup_datetime",
                            "tpep_dropoff_datetime": "dropoff_datetime"})
    df["pickup_datetime"] = df["pickup_datetime"].dt.strftime("%Y-%m-%d %H:%M:%S")
    df["dropoff_datetime"] = df["dropoff_datetime"].dt.strftime("%Y-%m-%d %H:%M:%S")
    print(f"ردیف قبل: {n0:,} | بعد از پاک‌سازی: {len(df):,} | حذف‌شده: {n0-len(df):,} ({(n0-len(df))/n0:.1%})")
    return df.reset_index(drop=True)

src = df_raw if SAMPLE_ROWS is None else df_raw.head(SAMPLE_ROWS)
df = clean_trips(src, MONTH)
df.head()

In [ ]:
clean_csv = BASE / "data" / "yellow_clean.csv"
clean_pq = BASE / "data" / "yellow_clean.parquet"
df.to_csv(clean_csv, index=False)            # با هدر؛ Mapper خط اول را رد می‌کند
df.to_parquet(clean_pq, index=False)
print("ذخیره شد:", clean_csv, "-", "%.1f MB" % (clean_csv.stat().st_size / 1e6))
print("ترتیب ستون‌ها:", list(df.columns))

<div dir="rtl" style="text-align:right; line-height:2">

> **ترتیب ستون‌ها (مهم برای Mapper):**
> `0:pickup_datetime، 1:dropoff_datetime، 2:passenger_count، 3:trip_distance، 4:PULocationID، 5:DOLocationID، 6:fare_amount، 7:total_amount، 8:duration_min، 9:pickup_date`

</div>

<div dir="rtl" style="text-align:right; line-height:2">

---
# بخش ۳: راه‌اندازی Hadoop با Docker و بارگذاری داده در HDFS

**توضیح اجزا:**
- **NameNode:** «فهرست‌بردار» HDFS؛ می‌داند هر فایل به چه بلوک‌هایی تقسیم شده و کجاست.
- **DataNode:** بلوک‌های واقعی داده را ذخیره می‌کند.
- **ResourceManager / NodeManager (YARN):** مدیریت منابع و اجرای وظایف MapReduce.

برای سادگی از یک DataNode و `replication=1` استفاده می‌کنیم (در محیط واقعی معمولاً ۳).

پوشه‌های `hadoop/` و `data/` را به داخل کانتینر `namenode` وصل می‌کنیم تا اسکریپت‌ها و داده پاک‌شده در دسترس باشند.

</div>

In [ ]:
%%writefile taxi_project/hadoop/docker-compose.yml
x-hadoop-env: &hadoop-env
  CORE-SITE.XML_fs.default.name: "hdfs://namenode"
  CORE-SITE.XML_fs.defaultFS: "hdfs://namenode"
  HDFS-SITE.XML_dfs.namenode.rpc-address: "namenode:8020"
  HDFS-SITE.XML_dfs.replication: "1"
  MAPRED-SITE.XML_mapreduce.framework.name: "yarn"
  MAPRED-SITE.XML_yarn.app.mapreduce.am.env: "HADOOP_MAPRED_HOME=$$HADOOP_HOME"
  MAPRED-SITE.XML_mapreduce.map.env: "HADOOP_MAPRED_HOME=$$HADOOP_HOME"
  MAPRED-SITE.XML_mapreduce.reduce.env: "HADOOP_MAPRED_HOME=$$HADOOP_HOME"
  YARN-SITE.XML_yarn.resourcemanager.hostname: "resourcemanager"
  YARN-SITE.XML_yarn.nodemanager.pmem-check-enabled: "false"
  YARN-SITE.XML_yarn.nodemanager.vmem-check-enabled: "false"
  YARN-SITE.XML_yarn.nodemanager.delete.debug-delay-sec: "600"
  YARN-SITE.XML_yarn.nodemanager.aux-services: "mapreduce_shuffle"
  CAPACITY-SCHEDULER.XML_yarn.scheduler.capacity.maximum-applications: "10000"
  CAPACITY-SCHEDULER.XML_yarn.scheduler.capacity.maximum-am-resource-percent: "0.5"
  CAPACITY-SCHEDULER.XML_yarn.scheduler.capacity.resource-calculator: "org.apache.hadoop.yarn.util.resource.DefaultResourceCalculator"
  CAPACITY-SCHEDULER.XML_yarn.scheduler.capacity.root.queues: "default"
  CAPACITY-SCHEDULER.XML_yarn.scheduler.capacity.root.default.capacity: "100"
  CAPACITY-SCHEDULER.XML_yarn.scheduler.capacity.root.default.user-limit-factor: "1"
  CAPACITY-SCHEDULER.XML_yarn.scheduler.capacity.root.default.maximum-capacity: "100"
  CAPACITY-SCHEDULER.XML_yarn.scheduler.capacity.root.default.state: "RUNNING"
  CAPACITY-SCHEDULER.XML_yarn.scheduler.capacity.root.default.acl_submit_applications: "*"
  CAPACITY-SCHEDULER.XML_yarn.scheduler.capacity.root.default.acl_administer_queue: "*"

services:
  namenode:
    image: apache/hadoop:3.3.6
    container_name: namenode
    hostname: namenode
    command: ["hdfs", "namenode"]
    ports:
      - "9870:9870"
    environment:
      <<: *hadoop-env
      ENSURE_NAMENODE_DIR: "/tmp/hadoop-root/dfs/name"
    volumes:
      - ./:/work
      - ../data:/data
  datanode:
    image: apache/hadoop:3.3.6
    container_name: datanode
    command: ["hdfs", "datanode"]
    environment:
      <<: *hadoop-env
  resourcemanager:
    image: apache/hadoop:3.3.6
    container_name: resourcemanager
    hostname: resourcemanager
    command: ["yarn", "resourcemanager"]
    ports:
      - "8088:8088"
    environment:
      <<: *hadoop-env
  nodemanager:
    image: apache/hadoop:3.3.6
    container_name: nodemanager
    command: ["yarn", "nodemanager"]
    environment:
      <<: *hadoop-env


In [ ]:
%%bash
cd taxi_project/hadoop
docker compose up -d
sleep 25
docker compose ps

<div dir="rtl" style="text-align:right; line-height:2">

**بررسی سلامت:** پس از چند ثانیه، رابط وب NameNode روی <http://localhost:9870> و YARN روی <http://localhost:8088> باز می‌شود (برای گزارش از این صفحه‌ها اسکرین‌شات بگیرید).

**نصب Python داخل کانتینرها:** ایمیج `apache/hadoop:3.3.6` روی CentOS 7 ساخته شده و Python ندارد. چون CentOS 7 بازنشسته شده، مخازن پیش‌فرض yum دیگر جواب نمی‌دهند و باید به `vault.centos.org` هدایت شوند. سلول زیر این کار را خودکار انجام می‌دهد. (Python باید هم در `namenode` و هم در `nodemanager` باشد، چون Mapper و Reducer روی NodeManager اجرا می‌شوند.) این نصب تا زمانی که کانتینرها را با `down` پاک نکنید باقی می‌ماند.

</div>

In [ ]:
%%bash
for c in namenode nodemanager; do
  echo "== $c =="
  if docker exec $c python3 --version 2>/dev/null; then continue; fi
  echo "python3 نصب نیست؛ مخزن yum به vault.centos.org منتقل و نصب انجام می‌شود..."
  docker exec -u root $c bash -c "
    sed -i 's/mirror.centos.org/vault.centos.org/g' /etc/yum.repos.d/*.repo &&
    sed -i 's/^#.*baseurl=http/baseurl=http/g' /etc/yum.repos.d/*.repo &&
    sed -i 's/^mirrorlist=http/#mirrorlist=http/g' /etc/yum.repos.d/*.repo &&
    yum clean all &&
    yum install -y --disablerepo='epel*' python3"
  docker exec $c python3 --version
done
docker exec namenode hdfs dfsadmin -report | grep -i "live datanodes"

In [ ]:
%%bash
docker exec namenode hdfs dfsadmin -safemode wait
docker exec namenode hdfs dfs -mkdir -p /taxi/input
docker exec namenode hdfs dfs -put -f /data/yellow_clean.csv /taxi/input/
docker exec namenode hdfs dfs -ls -h /taxi/input
docker exec namenode hdfs fsck /taxi/input/yellow_clean.csv -files -blocks | tail -15

<div dir="rtl" style="text-align:right; line-height:2">

**توضیح:** `fsck` نشان می‌دهد فایل به چند **بلوک** (پیش‌فرض ۱۲۸ مگابایت) تقسیم شده است. هر بلوک یک Map Task جداگانه خواهد داشت؛ این همان ایده «انتقال کد به سمت داده» در Hadoop است.

</div>

<div dir="rtl" style="text-align:right; line-height:2">

---
# بخش ۴: پردازش دسته‌ای با Hadoop MapReduce

**هدف:** محاسبه تعداد سفرها در هر روز و میانگین مسافت سفر در همان روز.

### منطق MapReduce
- **Map:** هر خط CSV را می‌خواند و جفت `(تاریخ , "مسافت,1")` تولید می‌کند.
- **Shuffle & Sort:** هدوپ خودکار همه مقدارهای یک تاریخ را کنار هم گرد می‌آورد و مرتب می‌کند.
- **Reduce:** برای هر تاریخ، تعداد و جمع مسافت‌ها را جمع می‌زند و `میانگین = جمع مسافت / تعداد` را چاپ می‌کند.

```
خط CSV ──Mapper──▶ (2024-01-05, "2.3,1") ──Shuffle/Sort──▶ Reducer ──▶ 2024-01-05  101234  3.412
```

چون Mapper و Reducer به زبان **Python** هستند، از **Hadoop Streaming** برای اجرا استفاده می‌کنیم: هدوپ ورودی را از طریق stdin به برنامه می‌دهد و خروجی stdout را می‌خواند.

</div>

In [ ]:
%%writefile taxi_project/hadoop/task4/mapper.py
#!/usr/bin/env python3
"""Mapper: خروجی (pickup_date \t trip_distance,1)"""
import sys

for line in sys.stdin:
    f = line.strip().split(",")
    if len(f) < 10 or f[0] == "pickup_datetime":     # رد کردن هدر و خطوط ناقص
        continue
    try:
        dist = float(f[3])
    except ValueError:
        continue
    print(f"{f[9]}\t{dist},1")

In [ ]:
%%writefile taxi_project/hadoop/task4/reducer.py
#!/usr/bin/env python3
"""Reducer: ورودی بر اساس کلید مرتب است. خروجی (date \t count \t avg_distance)"""
import sys

current, count, total = None, 0, 0.0

def emit():
    print(f"{current}\t{count}\t{total / count:.3f}")

for line in sys.stdin:
    key, val = line.rstrip("\n").split("\t", 1)
    dist, c = val.split(",")
    if key != current:
        if current is not None:
            emit()
        current, count, total = key, 0, 0.0
    count += int(c)
    total += float(dist)

if current is not None:
    emit()

<div dir="rtl" style="text-align:right; line-height:2">

**تست محلی (قبل از اجرا روی کلاستر):** با pipe یونیکس رفتار هدوپ را شبیه‌سازی می‌کنیم: `cat | mapper | sort | reducer`. این روش استاندارد دیباگ Streaming است.

</div>

In [ ]:
%%bash
cd taxi_project
head -5000 data/yellow_clean.csv | python3 hadoop/task4/mapper.py | sort | python3 hadoop/task4/reducer.py

In [ ]:
import subprocess, time, re, io

JAR = "/opt/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.3.6.jar"
timings = {}

def docker_exec(args, check=True):
    return subprocess.run(["docker", "exec", "namenode"] + args, capture_output=True, text=True, check=check)

def run_job(name, args):
    """اجرای job، اندازه‌گیری زمان و استخراج شمارنده‌های مهم از لاگ"""
    t0 = time.time()
    p = docker_exec(args, check=False)
    timings[name] = round(time.time() - t0, 1)
    log = p.stderr + p.stdout
    if p.returncode != 0:
        print(log[-3000:])
        raise RuntimeError(f"Job {name} failed")
    keys = ["Map input records", "Map output records", "Combine input records",
            "Combine output records", "Reduce input records", "Reduce output records"]
    for line in log.splitlines():
        if any(k in line for k in keys) or "Launched map tasks" in line or "Launched reduce tasks" in line:
            print(line.strip())
    print(f"⏱  {name}: {timings[name]} ثانیه")

docker_exec(["hdfs", "dfs", "-rm", "-r", "-f", "/taxi/output/task4"], check=False)
run_job("Task4 - Python Streaming", [
    "hadoop", "jar", JAR,
    "-files", "file:///work/task4/mapper.py,file:///work/task4/reducer.py",
    "-mapper", "python3 mapper.py",
    "-reducer", "python3 reducer.py",
    "-input", "/taxi/input/yellow_clean.csv",
    "-output", "/taxi/output/task4"])

In [ ]:
out = docker_exec(["bash", "-c", "hdfs dfs -cat /taxi/output/task4/part-*"]).stdout
mr_daily = pd.read_csv(io.StringIO(out), sep="\t", names=["date", "trips", "avg_distance"])
mr_daily.to_csv(BASE / "results" / "task4_daily.csv", index=False)
print(mr_daily.head(10))

import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(pd.to_datetime(mr_daily["date"]), mr_daily["trips"], marker="o")
ax[0].set_title("Trips per day (MapReduce)"); ax[0].tick_params(axis="x", rotation=45)
ax[1].bar(pd.to_datetime(mr_daily["date"]), mr_daily["avg_distance"], color="tab:orange")
ax[1].set_title("Average trip distance per day (miles)"); ax[1].tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.show()

<div dir="rtl" style="text-align:right; line-height:2">

**اعتبارسنجی نتیجه:** مجموع سفرهای MapReduce باید دقیقاً برابر تعداد ردیف‌های داده پاک‌شده باشد.

</div>

In [ ]:
print("مجموع سفرها در MapReduce:", int(mr_daily["trips"].sum()))
print("تعداد ردیف‌های داده پاک‌شده:", len(df))
assert int(mr_daily["trips"].sum()) == len(df), "عدم تطابق!"
print("✅ نتیجه صحیح است")

<div dir="rtl" style="text-align:right; line-height:2">

---
# بخش ۵: Hadoop Streaming — توزیع مدت زمان سفرها و مقایسه با MapReduce بومی (Java)

**مسئله:** مدت سفرها را به بازه‌های ۵ دقیقه‌ای (۰–۵، ۵–۱۰، …) تقسیم کنیم و تعداد سفر در هر بازه را بشماریم (هیستوگرام توزیع).

**نکته جدید: Combiner.** چون Reducer ما فقط جمع می‌زند (خاصیت شرکت‌پذیری)، همان را به‌عنوان Combiner روی خروجی هر Mapper هم اجرا می‌کنیم تا حجم داده‌ای که از شبکه Shuffle می‌شود به شدت کم شود.

</div>

In [ ]:
%%writefile taxi_project/hadoop/task5/mapper.py
#!/usr/bin/env python3
"""Mapper: بازه ۵ دقیقه‌ای مدت سفر → (bucket \t 1)"""
import sys

for line in sys.stdin:
    f = line.strip().split(",")
    if len(f) < 10 or f[0] == "pickup_datetime":
        continue
    try:
        dur = float(f[8])
    except ValueError:
        continue
    lo = int(dur // 5) * 5
    print(f"{lo:03d}-{lo + 5:03d}\t1")

In [ ]:
%%writefile taxi_project/hadoop/task5/reducer.py
#!/usr/bin/env python3
"""Reducer / Combiner: جمع تعداد برای هر بازه"""
import sys

current, count = None, 0
for line in sys.stdin:
    key, val = line.rstrip("\n").split("\t")
    if key != current:
        if current is not None:
            print(f"{current}\t{count}")
        current, count = key, 0
    count += int(val)
if current is not None:
    print(f"{current}\t{count}")

In [ ]:
docker_exec(["hdfs", "dfs", "-rm", "-r", "-f", "/taxi/output/task5"], check=False)
run_job("Task5 - Streaming + Combiner (duration histogram)", [
    "hadoop", "jar", JAR,
    "-files", "file:///work/task5/mapper.py,file:///work/task5/reducer.py",
    "-mapper", "python3 mapper.py",
    "-combiner", "python3 reducer.py",
    "-reducer", "python3 reducer.py",
    "-input", "/taxi/input/yellow_clean.csv",
    "-output", "/taxi/output/task5"])

out = docker_exec(["bash", "-c", "hdfs dfs -cat /taxi/output/task5/part-*"]).stdout
dur_hist = pd.read_csv(io.StringIO(out), sep="\t", names=["bucket_min", "trips"]).sort_values("bucket_min")
dur_hist.to_csv(BASE / "results" / "task5_duration_hist.csv", index=False)

dur_hist.plot(x="bucket_min", y="trips", kind="bar", figsize=(14, 4), legend=False, color="tab:green")
plt.title("Trip duration distribution (5-minute buckets) — Hadoop Streaming")
plt.xlabel("Duration (min)"); plt.ylabel("Trips"); plt.tight_layout(); plt.show()

<div dir="rtl" style="text-align:right; line-height:2">

**در شمارنده‌های بالا** مقدار `Combine input records` و `Combine output records` را مقایسه کنید: Combiner چند میلیون رکورد را قبل از Shuffle به چند ده رکورد کاهش داده است. این مهم‌ترین بهینه‌سازی در MapReduce است.

### اجرای نسخه Java (MapReduce بومی) برای مقایسه
همان کار بخش ۴ (سفر روزانه و میانگین مسافت) را با Java می‌نویسیم تا مقایسه منصفانه باشد. (این بخش اختیاری است؛ اگر کانتینر `javac` نداشت، خطا نمایش داده می‌شود و می‌توانید از آن بگذرید.)

</div>

In [ ]:
%%writefile taxi_project/hadoop/task5/DailyTrips.java
import java.io.IOException;
import org.apache.hadoop.conf.Configuration;
import org.apache.hadoop.fs.Path;
import org.apache.hadoop.io.LongWritable;
import org.apache.hadoop.io.Text;
import org.apache.hadoop.mapreduce.Job;
import org.apache.hadoop.mapreduce.Mapper;
import org.apache.hadoop.mapreduce.Reducer;
import org.apache.hadoop.mapreduce.lib.input.FileInputFormat;
import org.apache.hadoop.mapreduce.lib.output.FileOutputFormat;

public class DailyTrips {
  public static class M extends Mapper<LongWritable, Text, Text, Text> {
    private final Text k = new Text();
    private final Text v = new Text();
    @Override
    protected void map(LongWritable key, Text value, Context ctx) throws IOException, InterruptedException {
      String[] f = value.toString().split(",");
      if (f.length < 10 || f[0].equals("pickup_datetime")) return;
      try { Double.parseDouble(f[3]); } catch (NumberFormatException e) { return; }
      k.set(f[9]);
      v.set(f[3] + ",1");
      ctx.write(k, v);
    }
  }
  public static class R extends Reducer<Text, Text, Text, Text> {
    @Override
    protected void reduce(Text key, Iterable<Text> vals, Context ctx) throws IOException, InterruptedException {
      long cnt = 0; double tot = 0;
      for (Text t : vals) {
        String[] p = t.toString().split(",");
        tot += Double.parseDouble(p[0]);
        cnt += Long.parseLong(p[1]);
      }
      ctx.write(key, new Text(cnt + "\t" + String.format("%.3f", tot / cnt)));
    }
  }
  public static void main(String[] args) throws Exception {
    Job job = Job.getInstance(new Configuration(), "daily trips");
    job.setJarByClass(DailyTrips.class);
    job.setMapperClass(M.class);
    job.setReducerClass(R.class);
    job.setOutputKeyClass(Text.class);
    job.setOutputValueClass(Text.class);
    FileInputFormat.addInputPath(job, new Path(args[0]));
    FileOutputFormat.setOutputPath(job, new Path(args[1]));
    System.exit(job.waitForCompletion(true) ? 0 : 1);
  }
}

<div dir="rtl" style="text-align:right; line-height:2">

**نصب JDK (فقط برای کامپایل Java):** ایمیج هدوپ فقط JRE دارد و `javac` ندارد. سلول زیر JDK 8 را روی `namenode` نصب می‌کند (با Hadoop 3.3.6 سازگار است). اگر نمی‌خواهید بخش Java را اجرا کنید، از این سلول و سلول بعدی بگذرید.

</div>

In [ ]:
%%bash
docker exec namenode javac -version 2>&1 || docker exec -u root namenode bash -c "yum install -y --disablerepo='epel*' java-1.8.0-openjdk-devel"
docker exec namenode javac -version

In [ ]:
try:
    docker_exec(["bash", "-c",
        "cd /work/task5 && rm -rf classes && mkdir classes && "
        "javac -classpath $(hadoop classpath) -d classes DailyTrips.java && "
        "jar cf dailytrips.jar -C classes ."])
    docker_exec(["hdfs", "dfs", "-rm", "-r", "-f", "/taxi/output/task4_java"], check=False)
    run_job("Task4 - Java native", [
        "hadoop", "jar", "/work/task5/dailytrips.jar", "DailyTrips",
        "/taxi/input/yellow_clean.csv", "/taxi/output/task4_java"])
except Exception as e:
    print("⚠️ اجرای Java ممکن نشد (احتمالاً JDK/javac در کانتینر نیست):", str(e)[:300])

<div dir="rtl" style="text-align:right; line-height:2">

**بررسی یکسان بودن خروجی Java و Python:** هر دو job یک کار را انجام می‌دهند، پس خروجی‌شان باید بایت‌به‌بایت یکسان باشد. (فقط اگر Java اجرا شده باشد.)

</div>

In [ ]:
%%bash
docker exec namenode bash -c "hdfs dfs -cat /taxi/output/task4/part-* | sort > /tmp/py.txt; hdfs dfs -cat /taxi/output/task4_java/part-* | sort > /tmp/java.txt; diff /tmp/py.txt /tmp/java.txt && echo IDENTICAL"

<div dir="rtl" style="text-align:right; line-height:2">

**آزمایش اختیاری: اثر خالص Combiner در زمان.** همان job هیستوگرام را بدون Combiner اجرا می‌کنیم. این بار `Reduce input records` باید حدود ۲٫۷ میلیون باشد (به‌جای ۷۲). زمان این اجرا را با اجرای قبلی (با Combiner) مقایسه کنید؛ چون هر دو یک کار می‌کنند، مقایسه عادلانه است.

</div>

In [ ]:
docker_exec(["hdfs", "dfs", "-rm", "-r", "-f", "/taxi/output/task5_nocomb"], check=False)
run_job("Task5 - Streaming NO Combiner", [
    "hadoop", "jar", JAR,
    "-files", "file:///work/task5/mapper.py,file:///work/task5/reducer.py",
    "-mapper", "python3 mapper.py",
    "-reducer", "python3 reducer.py",
    "-input", "/taxi/input/yellow_clean.csv",
    "-output", "/taxi/output/task5_nocomb"])

In [ ]:
comparison = pd.DataFrame({"job": list(timings.keys()), "seconds": list(timings.values())})
print(comparison.to_string(index=False))

<div dir="rtl" style="text-align:right; line-height:2">

### مقایسه عملکرد و سهولت استفاده

| معیار | Hadoop Streaming (Python) | MapReduce بومی (Java) |
|-------|---------------------------|------------------------|
| **سرعت توسعه** | بسیار سریع؛ ۱۰–۲۰ خط Python و تست با pipe | کندتر؛ نیاز به کلاس، کامپایل، jar و مدیریت نوع‌ها (Writable) |
| **اشکال‌زدایی** | ساده؛ بدون کلاستر هم تست می‌شود | سخت‌تر؛ نیاز به build و اجرای واقعی |
| **سرعت اجرا** | معمولاً کندتر؛ سربار راه‌اندازی مفسر Python و تبدیل متن ↔ بایت برای هر رکورد | معمولاً سریع‌تر؛ اجرا مستقیم در JVM و Writableهای باینری |
| **قابلیت‌ها** | محدود به stdin/stdout، فقط متن | دسترسی کامل به API: Partitioner سفارشی، InputFormat، Counters و … |
| **کتابخانه‌ها** | کل اکوسیستم Python | کتابخانه‌های جاوا |
| **مناسب برای** | نمونه‌سازی، تحلیل‌های ساده، تیم‌های Python | سیستم‌های تولیدی و کارهای بسیار حجیم |

**نکته درباره اعتبار مقایسه:** فقط دو job با کار یکسان قابل مقایسه‌اند: «Python Streaming» و «Java native» (هر دو تعداد و میانگین مسافت روزانه). job هیستوگرام کار دیگری می‌کند و فقط با نسخه «بدون Combiner» خودش قابل مقایسه است. هر زمان یک اندازه‌گیری تکی است و بهتر است چند بار تکرار شود.

**نتیجه‌گیری:** برای یادگیری و پروتوتایپ Streaming راحت‌تر است؛ برای عملکرد حداکثری Java برتری دارد. (اعداد دقیق جدول زمان‌بندی بالا را در گزارش بیاورید؛ روی داده کوچک سربار راه‌اندازی job غالب است و تفاوت واقعی کمتر دیده می‌شود.) امروزه Spark هر دو را عملاً کنار زده است، که در بخش بعد می‌بینیم.

</div>

<div dir="rtl" style="text-align:right; line-height:2">

---
# بخش ۶: راه‌اندازی Spark و تحلیل داده با PySpark

**Spark چه مزیتی دارد؟** برخلاف MapReduce که هر مرحله را روی دیسک می‌نویسد، Spark داده را **در حافظه** نگه می‌دارد و با DataFrame API کد بسیار کوتاه‌تری لازم دارد. معمولاً ۱۰ تا ۱۰۰ برابر سریع‌تر از MapReduce است.

ما Spark را در **حالت Local** (روی همه هسته‌های همین سیستم) اجرا می‌کنیم؛ همین کد روی کلاستر واقعی هم بدون تغییر عمده کار می‌کند. پکیج اتصال Kafka را هم از همین ابتدا به Session اضافه می‌کنیم تا در بخش ۷ نیاز به ساخت دوباره نباشد.

> نیاز به **Java 11 یا 17**: با `java -version` بررسی کنید.

</div>

In [ ]:
%%bash
java -version 2>&1 | head -1

In [ ]:
from pyspark.sql import SparkSession, functions as F, types as T

spark = (SparkSession.builder
         .master("local[*]")
         .appName("TaxiAnalytics")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version, "| UI:", spark.sparkContext.uiWebUrl)

In [ ]:
schema = T.StructType([
    T.StructField("pickup_datetime", T.TimestampType()),
    T.StructField("dropoff_datetime", T.TimestampType()),
    T.StructField("passenger_count", T.IntegerType()),
    T.StructField("trip_distance", T.DoubleType()),
    T.StructField("PULocationID", T.IntegerType()),
    T.StructField("DOLocationID", T.IntegerType()),
    T.StructField("fare_amount", T.DoubleType()),
    T.StructField("total_amount", T.DoubleType()),
    T.StructField("duration_min", T.DoubleType()),
    T.StructField("pickup_date", T.StringType()),
])
trips = spark.read.csv(str((BASE / "data" / "yellow_clean.csv").resolve()), header=True, schema=schema).cache()
print("تعداد سفرها:", trips.count())
trips.printSchema()
trips.show(5)

<div dir="rtl" style="text-align:right; line-height:2">

> **توضیح:** در این پروژه Spark فایل را از دیسک محلی می‌خواند. اگر Spark را روی همان شبکه Docker هدوپ اجرا کنید، می‌توانید به‌جای آن از `hdfs://namenode:8020/taxi/input/yellow_clean.csv` بخوانید.

### تحلیل ۱: فیلتر کردن سفرها
سفرهای «طولانی ولی کوتاه‌مدت» (بیش از ۵ مایل در کمتر از ۱۵ دقیقه، مثلاً سفر فرودگاهی سریع) و همچنین سفرهای با بیش از ۳ مسافر:

</div>

In [ ]:
fast_long = trips.filter((F.col("trip_distance") > 5) & (F.col("duration_min") < 15))
print("سفرهای >5 مایل در <15 دقیقه:", fast_long.count())
group_trips = trips.filter(F.col("passenger_count") > 3)
print("سفرهای بیش از ۳ مسافر:", group_trips.count())
fast_long.select("pickup_datetime", "trip_distance", "duration_min", "PULocationID").show(5)

<div dir="rtl" style="text-align:right; line-height:2">

### تحلیل ۲: میانگین مدت سفر بر حسب مکان سوار شدن
برای خوانا شدن، شناسه مناطق (`PULocationID`) را با فایل **Taxi Zone Lookup** به نام منطقه تبدیل می‌کنیم (Join). فقط مناطقی با حداقل ۵۰۰ سفر را نگه می‌داریم تا میانگین‌ها معنادار باشند.

</div>

In [ ]:
zones_pd = pd.read_csv("https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv")
zones_pd.to_csv(BASE / "data" / "taxi_zone_lookup.csv", index=False)
zones = spark.createDataFrame(zones_pd[["LocationID", "Borough", "Zone"]].fillna("Unknown"))

avg_by_loc = (trips.groupBy("PULocationID")
              .agg(F.count("*").alias("trips"),
                   F.round(F.avg("duration_min"), 2).alias("avg_duration_min"),
                   F.round(F.avg("trip_distance"), 2).alias("avg_distance"))
              .filter("trips >= 500")
              .join(zones, F.col("PULocationID") == F.col("LocationID"), "left")
              .select("PULocationID", "Borough", "Zone", "trips", "avg_duration_min", "avg_distance"))

print("۱۰ منطقه با بیشترین میانگین مدت سفر:")
avg_by_loc.orderBy(F.desc("avg_duration_min")).show(10, truncate=False)
print("۱۰ منطقه پرترافیک (بیشترین تعداد سفر):")
avg_by_loc.orderBy(F.desc("trips")).show(10, truncate=False)

<div dir="rtl" style="text-align:right; line-height:2">

### تحلیل ۳: سرعت متوسط بر حسب ساعت روز و اعتبارسنجی با نتیجه MapReduce
- **سرعت متوسط** = مسافت ÷ (مدت به ساعت). ساعات شلوغ باید سرعت کمتری داشته باشند.
- همچنین تعداد سفر روزانه را با Spark حساب می‌کنیم و با نتیجه MapReduce مقایسه می‌کنیم (باید یکسان باشد).

</div>

In [ ]:
hourly = (trips.withColumn("hour", F.hour("pickup_datetime"))
          .withColumn("mph", F.col("trip_distance") / (F.col("duration_min") / 60))
          .groupBy("hour")
          .agg(F.count("*").alias("trips"), F.round(F.avg("mph"), 2).alias("avg_speed_mph"))
          .orderBy("hour"))
hourly_pd = hourly.toPandas()

fig, ax1 = plt.subplots(figsize=(12, 4))
ax1.bar(hourly_pd["hour"], hourly_pd["trips"], alpha=0.6)
ax1.set_xlabel("Hour of day"); ax1.set_ylabel("Trips")
ax2 = ax1.twinx(); ax2.plot(hourly_pd["hour"], hourly_pd["avg_speed_mph"], color="red", marker="o")
ax2.set_ylabel("Avg speed (mph)")
plt.title("Trips and average speed by hour"); plt.show()

# مقایسه با MapReduce
spark_daily = (trips.groupBy("pickup_date")
               .agg(F.count("*").alias("trips"), F.round(F.avg("trip_distance"), 3).alias("avg_distance"))
               .orderBy("pickup_date").toPandas())
merged = spark_daily.merge(mr_daily, left_on="pickup_date", right_on="date", suffixes=("_spark", "_mr"))
print("روزهای یکسان در تعداد سفر:", (merged["trips_spark"] == merged["trips_mr"]).all())
print("بیشترین اختلاف میانگین مسافت:", (merged["avg_distance_spark"] - merged["avg_distance_mr"]).abs().max())

In [ ]:
# ذخیره نتایج Spark برای گزارش
avg_by_loc.toPandas().to_csv(BASE / "results" / "task6_avg_by_location.csv", index=False)
hourly_pd.to_csv(BASE / "results" / "task6_hourly.csv", index=False)

<div dir="rtl" style="text-align:right; line-height:2">

**آزادسازی منابع:** Hadoop دیگر لازم نیست. قبل از رفتن به بخش ۷ از صفحه YARN (<http://localhost:8088>) و HDFS (<http://localhost:9870>) اسکرین‌شات بگیرید، بعد کانتینرها را فقط متوقف کنید (داده HDFS حفظ می‌شود، برخلاف `down -v`).

</div>

In [ ]:
%%bash
cd taxi_project/hadoop
docker compose stop

<div dir="rtl" style="text-align:right; line-height:2">

---
# بخش ۷: پردازش بلادرنگ با Kafka و Spark Structured Streaming

**معماری:**

```
Producer (شبیه‌ساز سفر زنده) ──▶ Kafka topic: taxi-trips ──▶ Spark Structured Streaming ──▶ جداول حافظه ──▶ داشبورد
```

- **Kafka:** صف پیام توزیع‌شده. Producer پیام می‌نویسد و Consumer می‌خواند، بدون وابستگی به هم (Decoupling).
- **Topic:** کانال نام‌دار پیام‌ها؛ اینجا `taxi-trips`.
- **Structured Streaming:** استریم را مثل یک جدول بی‌نهایت می‌بیند که ردیف‌هایش مدام اضافه می‌شود؛ همان DataFrame API دسته‌ای قابل استفاده است.

از Kafka در حالت **KRaft** (بدون ZooKeeper) و تک‌گره استفاده می‌کنیم.

</div>

In [ ]:
%%writefile taxi_project/kafka/docker-compose.yml
services:
  kafka:
    image: apache/kafka:3.7.0
    container_name: kafka
    ports:
      - "9092:9092"
    environment:
      KAFKA_NODE_ID: 1
      KAFKA_PROCESS_ROLES: broker,controller
      KAFKA_LISTENERS: PLAINTEXT://:9092,CONTROLLER://:9093
      KAFKA_ADVERTISED_LISTENERS: PLAINTEXT://localhost:9092
      KAFKA_CONTROLLER_LISTENER_NAMES: CONTROLLER
      KAFKA_LISTENER_SECURITY_PROTOCOL_MAP: CONTROLLER:PLAINTEXT,PLAINTEXT:PLAINTEXT
      KAFKA_CONTROLLER_QUORUM_VOTERS: 1@localhost:9093
      KAFKA_OFFSETS_TOPIC_REPLICATION_FACTOR: 1
      KAFKA_TRANSACTION_STATE_LOG_REPLICATION_FACTOR: 1
      KAFKA_TRANSACTION_STATE_LOG_MIN_ISR: 1
      KAFKA_GROUP_INITIAL_REBALANCE_DELAY_MS: 0

In [ ]:
%%bash
cd taxi_project/kafka
docker compose up -d
sleep 15
docker exec kafka /opt/kafka/bin/kafka-topics.sh --create --if-not-exists \
  --topic taxi-trips --partitions 3 --replication-factor 1 --bootstrap-server localhost:9092
docker exec kafka /opt/kafka/bin/kafka-topics.sh --describe --topic taxi-trips --bootstrap-server localhost:9092

<div dir="rtl" style="text-align:right; line-height:2">

### Producer: شبیه‌ساز جریان زنده سفرها
Producer تعدادی سفر تصادفی از داده پاک‌شده برمی‌دارد و با نرخ مشخص (پیش‌فرض ۱۰۰ پیام در ثانیه) به‌صورت **JSON** به Kafka می‌فرستد؛ مثل اینکه تاکسی‌ها همین الان سفر را تمام می‌کنند. آن را به‌صورت فرایند جدا اجرا می‌کنیم تا همزمان با Spark کار کند.

</div>

In [ ]:
%%writefile taxi_project/kafka/producer.py
import argparse, json, time
import pandas as pd
from kafka import KafkaProducer

ap = argparse.ArgumentParser()
ap.add_argument("--csv", required=True)
ap.add_argument("--n", type=int, default=9000)
ap.add_argument("--rate", type=float, default=100)     # پیام در ثانیه
ap.add_argument("--topic", default="taxi-trips")
a = ap.parse_args()

cols = ["pickup_datetime", "passenger_count", "trip_distance", "PULocationID",
        "DOLocationID", "fare_amount", "duration_min"]
df = pd.read_csv(a.csv, usecols=cols, nrows=300000)
df = df.sample(n=min(a.n, len(df)), random_state=42)

producer = KafkaProducer(bootstrap_servers="localhost:9092",
                         value_serializer=lambda v: json.dumps(v).encode("utf-8"))
for i, rec in enumerate(df.to_dict("records"), 1):
    producer.send(a.topic, rec)
    time.sleep(1.0 / a.rate)
    if i % 1000 == 0:
        print(f"sent {i}", flush=True)
producer.flush()
print("done", flush=True)

<div dir="rtl" style="text-align:right; line-height:2">

### Consumer: برنامه Spark Streaming
سه کوئری استریم همزمان اجرا می‌شود و نتیجه هر کدام در یک **جدول حافظه** (Memory Sink) ذخیره می‌شود تا داشبورد در بخش ۸ آن‌ها را با SQL بخواند:

1. `window_stats`: در هر **پنجره زمانی ۱۰ ثانیه‌ای**، تعداد سفر، میانگین مسافت و میانگین مدت سفر.
2. `dist_hist`: توزیع مسافت سفرها (دسته‌های ۱ مایلی).
3. `zone_stats`: تعداد سفر و میانگین مدت به تفکیک منطقه سوار شدن.

زمان رویداد را از **timestamp پیام Kafka** می‌گیریم. حالت خروجی `complete` یعنی در هر Trigger کل جدول نتیجه بازنویسی می‌شود (برای جدول‌های کوچک مناسب است).

</div>

In [ ]:
json_schema = T.StructType([
    T.StructField("pickup_datetime", T.StringType()),
    T.StructField("passenger_count", T.IntegerType()),
    T.StructField("trip_distance", T.DoubleType()),
    T.StructField("PULocationID", T.IntegerType()),
    T.StructField("DOLocationID", T.IntegerType()),
    T.StructField("fare_amount", T.DoubleType()),
    T.StructField("duration_min", T.DoubleType()),
])

raw = (spark.readStream.format("kafka")
       .option("kafka.bootstrap.servers", "localhost:9092")
       .option("subscribe", "taxi-trips")
       .option("startingOffsets", "earliest")
       .load())

stream = (raw.select(F.from_json(F.col("value").cast("string"), json_schema).alias("t"),
                     F.col("timestamp").alias("event_time"))
          .select("t.*", "event_time"))

def start(df_agg, name):
    return (df_agg.writeStream.outputMode("complete").format("memory")
            .queryName(name).trigger(processingTime="5 seconds").start())

q_window = start(stream.groupBy(F.window("event_time", "10 seconds"))
                 .agg(F.count("*").alias("trips"),
                      F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
                      F.round(F.avg("duration_min"), 2).alias("avg_duration")), "window_stats")

q_hist = start(stream.filter("trip_distance < 20")
               .withColumn("dist_bucket", F.floor("trip_distance"))
               .groupBy("dist_bucket").count(), "dist_hist")

q_zone = start(stream.groupBy("PULocationID")
               .agg(F.count("*").alias("trips"), F.round(F.avg("duration_min"), 2).alias("avg_duration")),
               "zone_stats")

print("کوئری‌های فعال:", [q.name for q in spark.streams.active])

In [ ]:
import sys, subprocess
producer_proc = subprocess.Popen(
    [sys.executable, str(BASE / "kafka" / "producer.py"),
     "--csv", str(BASE / "data" / "yellow_clean.csv"), "--n", "9000", "--rate", "100"])
print("Producer شروع شد (PID %d). حدود ۹۰ ثانیه طول می‌کشد." % producer_proc.pid)

In [ ]:
time.sleep(15)
spark.sql("SELECT window.start AS start, trips, avg_distance, avg_duration FROM window_stats ORDER BY start DESC").show(5, truncate=False)
spark.sql("SELECT * FROM dist_hist ORDER BY dist_bucket").show(5)

<div dir="rtl" style="text-align:right; line-height:2">

اگر ردیف‌هایی با افزایش مداوم `trips` می‌بینید، خط لوله **Kafka → Spark Streaming** کار می‌کند. حالا آن را در داشبورد تصویری می‌کنیم.

</div>

<div dir="rtl" style="text-align:right; line-height:2">

---
# بخش ۸: داشبورد تعاملی

دو بخش داریم:
1. **داشبورد دسته‌ای (Batch):** شاخص‌های کلیدی کل ماه از داده پاک‌شده.
2. **داشبورد زنده (Real-time):** نمودارهایی که هر ۵ ثانیه از جدول‌های استریم بازخوانی می‌شوند.

> اگر Producer هنوز در حال ارسال است، همین حالا بخش زنده را اجرا کنید. اگر تمام شده بود، سلول ارسال (بخش ۷) را دوباره اجرا کنید.

</div>

In [ ]:
d = pd.read_parquet(BASE / "data" / "yellow_clean.parquet")
d["pickup_datetime"] = pd.to_datetime(d["pickup_datetime"])
d["hour"] = d["pickup_datetime"].dt.hour

fig, ax = plt.subplots(2, 3, figsize=(18, 9))
ax[0, 0].hist(d["trip_distance"].clip(upper=20), bins=40, color="tab:blue")
ax[0, 0].set_title("Trip distance distribution (miles, clipped at 20)")
ax[0, 1].hist(d["duration_min"].clip(upper=60), bins=40, color="tab:green")
ax[0, 1].set_title("Trip duration distribution (min, clipped at 60)")
d.groupby("hour").size().plot(kind="bar", ax=ax[0, 2], color="tab:purple"); ax[0, 2].set_title("Trips by hour")
d.groupby("pickup_date").size().plot(ax=ax[1, 0], marker="o"); ax[1, 0].set_title("Trips per day")
ax[1, 0].tick_params(axis="x", rotation=60)
top = d["PULocationID"].value_counts().head(10).rename_axis("LocationID").reset_index(name="trips")
top = top.merge(zones_pd[["LocationID", "Zone"]], on="LocationID", how="left")
ax[1, 1].barh(top["Zone"].fillna(top["LocationID"].astype(str)), top["trips"], color="tab:orange")
ax[1, 1].invert_yaxis(); ax[1, 1].set_title("Top 10 pickup zones")
ax[1, 2].scatter(d["trip_distance"].sample(5000, random_state=1).clip(upper=30),
                 d["fare_amount"].sample(5000, random_state=1).clip(upper=150), s=4, alpha=0.4)
ax[1, 2].set_title("Fare vs distance (sample)"); ax[1, 2].set_xlabel("miles"); ax[1, 2].set_ylabel("USD")
plt.tight_layout(); plt.show()

kpi = {"Total trips": f"{len(d):,}", "Avg distance (mi)": round(d.trip_distance.mean(), 2),
       "Avg duration (min)": round(d.duration_min.mean(), 1), "Avg fare ($)": round(d.fare_amount.mean(), 2),
       "Busiest hour": int(d.groupby("hour").size().idxmax())}
pd.Series(kpi).to_frame("value")

<div dir="rtl" style="text-align:right; line-height:2">

### داشبورد زنده
هر ۵ ثانیه، جدول‌های Memory Sink با SQL خوانده و نمودارها دوباره کشیده می‌شوند.

**نکات مهم:**
- جدول‌های استریم **تجمعی‌اند** (حالت `complete`)؛ اگر Producer را چند بار اجرا کنید، شمارنده کل مجموع همه اجراهاست (مثلاً ۴ اجرا = ۳۶٬۰۰۰).
- دو نمودار زمانی فقط `last_minutes` دقیقه آخر را نشان می‌دهند تا فاصله زمانی بین اجراها خط صاف و بلند نسازد.
- آخرین نقطه پنجره زمانی معمولاً ناقص است (پنجره هنوز پر نشده) و افت آن تغییر واقعی تقاضا نیست.
- ترتیب اجرا: اول سلول Producer، بلافاصله بعد از آن همین سلول.

</div>

In [ ]:
from IPython.display import clear_output
zone_names = dict(zip(zones_pd["LocationID"], zones_pd["Zone"]))

def live_dashboard(iterations=18, refresh=5, last_minutes=3):
    for i in range(iterations):
        w = spark.sql("SELECT window.start AS start, trips, avg_distance, avg_duration FROM window_stats ORDER BY start").toPandas()
        h = spark.sql("SELECT * FROM dist_hist ORDER BY dist_bucket").toPandas()
        z = spark.sql("SELECT * FROM zone_stats ORDER BY trips DESC LIMIT 10").toPandas()

        # شمارنده کل از همه پنجره‌ها، قبل از فیلتر
        total = int(w["trips"].sum()) if len(w) else 0
        # فقط چند دقیقه آخر را برای نمودارهای زمانی نگه می‌داریم
        if len(w):
            w = w[w["start"] >= w["start"].max() - pd.Timedelta(minutes=last_minutes)]

        clear_output(wait=True)
        fig, ax = plt.subplots(2, 2, figsize=(15, 8))
        if len(w):
            ax[0, 0].plot(w["start"], w["trips"], marker="o")
            ax[0, 0].set_title("Trips per 10-second window")
            ax[0, 1].plot(w["start"], w["avg_duration"], marker="o", color="tab:green")
            ax[0, 1].set_title("Real-time avg trip duration (min)")
            for a in (ax[0, 0], ax[0, 1]):
                a.tick_params(axis="x", rotation=45)
        if len(h):
            ax[1, 0].bar(h["dist_bucket"], h["count"], color="tab:blue")
            ax[1, 0].set_title("Trip distance distribution (streamed)")
            ax[1, 0].set_xlabel("miles")
        if len(z):
            ax[1, 1].barh([zone_names.get(int(k), str(k)) for k in z["PULocationID"]], z["trips"], color="tab:orange")
            ax[1, 1].invert_yaxis()
            ax[1, 1].set_title("Top pickup zones (streamed)")
        fig.suptitle(f"LIVE — refresh {i+1}/{iterations} — total streamed trips: {total:,}", fontsize=14)
        plt.tight_layout()
        plt.show()
        time.sleep(refresh)

live_dashboard()

In [ ]:
# پایان: توقف استریم‌ها و Producer
for q in spark.streams.active:
    q.stop()
if producer_proc.poll() is None:
    producer_proc.terminate()
print("همه کوئری‌ها متوقف شدند.")

<div dir="rtl" style="text-align:right; line-height:2">

---
# بخش ۹: گزارش نهایی و ارائه

### ۹.۱ خلاصه نتایج (برای گزارش)

</div>

In [ ]:
pd.set_option("display.max_colwidth", None)
summary = pd.DataFrame({
    "مرحله": ["۲. پاک‌سازی", "۳. HDFS", "۴. MapReduce (Python Streaming)",
              "۴. MapReduce (Java، مقایسه)", "۵. Streaming + Combiner",
              "۶. Spark", "۷. Kafka + Spark Streaming"],
    "نتیجه": ["۲٬۹۶۴٬۶۲۴ ردیف خام ← ۲٬۷۱۱٬۷۹۴ ردیف معتبر (حذف ۸٫۵٪)",
              "فایل ۲۱۹٫۸ مگابایت، ۲ بلوک، وضعیت HEALTHY",
              "۷۴٫۸ ثانیه؛ ۳۱ روز؛ ۲٫۷ میلیون رکورد Shuffle",
              "۴۳٫۰ ثانیه؛ خروجی یکسان با Python (diff)",
              "۳۸٫۵ ثانیه؛ رکورد Shuffle از ۲٫۷ میلیون به ۷۲",
              "تعداد ردیف و ۳۱ روز دقیقاً مطابق MapReduce",
              "۹۰۰۰ پیام ارسال‌شده = ۹۰۰۰ دریافت‌شده"],
})
summary

<div dir="rtl" style="text-align:right; line-height:2">

### ۹.۲ ساختار پیشنهادی گزارش

۱. **مقدمه و هدف پروژه** (معماری کلی؛ می‌توانید دیاگرام بخش اول نوت‌بوک را بیاورید)
۲. **پنج V در بیگ‌دیتا** (بخش ۱)
۳. **دریافت و پاک‌سازی داده** — جدول قبل/بعد پاک‌سازی و معیارها
۴. **محیط Hadoop** — فایل compose، اسکرین‌شات `localhost:9870` و `localhost:8088`، خروجی `hdfs dfs -ls`
۵. **MapReduce** — کد Mapper/Reducer، توضیح منطق، لاگ اجرا و نمودار روزانه
۶. **Hadoop Streaming + Combiner** — هیستوگرام و جدول مقایسه با Java
۷. **Spark** — کدها و خروجی تحلیل‌ها (فیلتر، میانگین بر حسب منطقه، سرعت ساعتی)
۸. **Kafka + Spark Streaming** — معماری، Producer، کوئری‌های استریم
۹. **داشبورد** — اسکرین‌شات داشبورد دسته‌ای و زنده
۱۰. **نتیجه‌گیری، چالش‌ها و کارهای آینده**

> **اسکرین‌شات‌هایی که باید بگیرید:** HDFS UI، YARN UI (job تمام‌شده)، Spark UI (`http://localhost:4040` هنگام اجرای استریم)، خروجی `fsck`، نمودارهای هر دو داشبورد.

### ۹.۳ چالش‌های رایج و راه‌حل‌ها

| چالش | راه‌حل |
|------|--------|
| داده بزرگ و کند شدن MapReduce روی لپ‌تاپ | متغیر `SAMPLE_ROWS` را کم کنید یا فقط یک ماه را بردارید |
| `python3: not found` در job هدوپ | Python را داخل `namenode` و `nodemanager` نصب کنید (سلول بخش ۳) |
| NameNode در Safe Mode است | `hdfs dfsadmin -safemode wait` یا چند ثانیه صبر کنید |
| Spark به Kafka وصل نمی‌شود | پورت `9092` آزاد باشد و `KAFKA_ADVERTISED_LISTENERS` روی `localhost` باشد؛ اینترنت برای دانلود پکیج کافکا لازم است |
| نمودار خالی در داشبورد زنده | Producer در حال اجرا باشد؛ چند ثانیه صبر کنید تا اولین Trigger اجرا شود |
| داده خراب (مسافت صفر، تاریخ قدیمی) | قوانین پاک‌سازی بخش ۲ (Veracity) |
| `docker compose up` خطای `unexpected character "-"` | تنظیمات هدوپ را در `environment` خود compose بگذارید (نه `env_file`) و `$HADOOP_HOME` را به‌صورت `$$HADOOP_HOME` بنویسید |
| `yum install python3` شکست می‌خورد (CentOS 7 بازنشسته) | مخزن را به `vault.centos.org` هدایت کنید (سلول بخش ۳) |
| `javac: not found` (کد خروج ۱۲۷) | JDK را فقط روی `namenode` نصب کنید (سلول بخش ۵) |
| نمودار زنده خط صاف و بلند دارد | فقط چند دقیقه آخر نمایش داده شود (`last_minutes`) |

### ۹.۴ ایده‌های ارائه (۱۰ اسلاید)

1. عنوان و هدف پروژه
2. پنج V و ارتباط با داده تاکسی
3. معماری کلی خط لوله (دیاگرام)
4. پاک‌سازی داده: قبل/بعد
5. HDFS و Hadoop روی Docker
6. MapReduce: منطق Map → Shuffle → Reduce و نتیجه
7. Streaming vs Java + نقش Combiner
8. Spark: تحلیل‌ها و سرعت نسبت به MapReduce
9. Kafka + Spark Streaming + داشبورد زنده (دمو)
10. درس‌ها، چالش‌ها و کارهای آینده

### ۹.۵ درس‌های آموخته
- **انتخاب ابزار بر اساس نیاز:** MapReduce پایه‌ای برای فهم مفاهیم است، Spark برای کار روزمره و Kafka برای داده زنده.
- **کیفیت داده مقدم بر هر تحلیل است** (Veracity).
- **Combiner و Partitioning** تأثیر زیادی بر کارایی دارند.
- **تست محلی با pipe** پیش از اجرا روی کلاستر، زمان زیادی صرفه‌جویی می‌کند.
- **کارهای آینده:** افزودن HDFS به‌عنوان منبع Spark، پایش با Grafana، ذخیره نتایج استریم در پایگاه‌داده یا Parquet و پیش‌بینی تقاضا با Spark MLlib.

### پاک‌سازی منابع (پس از پایان)

</div>

In [ ]:
%%bash
cd taxi_project/kafka && docker compose down -v
cd ../hadoop && docker compose down -v

In [ ]:
spark.stop()